# SEED-IV feature tuning monitor (diagnostic only)

This notebook provides one generic training loop for `CMRD`, `RJSD`, and `DE`. The model is supplied through a replaceable `build_model(...)` factory, so the data pipeline and epoch monitoring do not depend on a particular architecture.

> **Target-peeking warning:** the held-out target subject is evaluated and printed every epoch for convenient diagnosis. Any hyperparameter or epoch chosen after inspecting these target curves is contaminated by target-domain information and must not be reported as a clean final result. Use the source-validation-selected training script for formal results.

In [1]:
from __future__ import annotations

import gc
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch import nn
from torch.utils.data import DataLoader


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src' / 'cmrd').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the CMRD repository')


ROOT = find_project_root(Path.cwd())
for path in (ROOT / 'src', ROOT / 'scripts'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import train_seediv_de_rjsd_ica as feature_pipeline
from cmrd.data.records import TrialSample
from cmrd.features.rd import normalize_histograms, transform_rd
from cmrd.models import PlainTransformer
from cmrd.training.engine import SequenceDataset, collate_sequences, fit_normalizer
from cmrd.training.metrics import classification_metrics
from cmrd.training.runtime import seed_everything

print('Project:', ROOT)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Project: C:\Users\Lin\Documents\Arbitruam\CMRD-Cute-Mew-Really-Delighting
PyTorch: 2.13.0.dev20260422+cu132
CUDA available: True
GPU: NVIDIA GeForce RTX 5080 Laptop GPU


## Parameters

Change `FEATURE` to `CMRD`, `RJSD`, or `DE`. Start with one target subject and a small epoch count; switch to all 15 subjects only after the notebook behaves as expected.

In [2]:
# Data / feature
FEATURE = 'RJSD'              # CMRD | RJSD | DE | FUSION (case-insensitive)
ALPHA = 0.5                   # Used by CMRD/FUSION only
DATA_ROOT = None              # None = auto-discover the completed signature cache
TARGET_SUBJECTS = [1]         # For all folds: list(range(1, 16))

# Training
EPOCHS = 100
TEST_EVERY = 10               # Evaluate source/target every N epochs
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
MINIMUM_LEARNING_RATE = 1e-6
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.2
GRADIENT_CLIP_NORM = 1.0
SEED = 42
DETERMINISTIC = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_WORKERS = 0               # Safest setting for Windows/Jupyter

# The default factory below uses these values. Replace the factory cell for another model.
MODEL_NAME = 'plain_transformer_example'
MODEL_CONFIG = {
    'd_model': 600,
    'nhead': 12,
    'layers': 6,
    'feedforward': 2400,
    'dropout': 0.2,
}

# Output
RUN_TAG = f"{FEATURE.lower()}_{MODEL_NAME}_all_source_every{TEST_EVERY}_diagnostic"
RESUME_COMPLETED_FOLDS = True
SAVE_FINAL_CHECKPOINT = True

FEATURE = feature_pipeline._feature_name(FEATURE)
assert FEATURE in feature_pipeline.FEATURES
assert all(1 <= subject <= 15 for subject in TARGET_SUBJECTS)
assert len(TARGET_SUBJECTS) == len(set(TARGET_SUBJECTS))
assert MODEL_CONFIG['d_model'] % MODEL_CONFIG['nhead'] == 0
assert EPOCHS > 0 and BATCH_SIZE > 0 and TEST_EVERY > 0
DEVICE_OBJ = torch.device(DEVICE)
RUN_DIR = ROOT / 'runs' / 'diagnostics' / 'seediv_feature_tuning' / RUN_TAG
RUN_DIR.mkdir(parents=True, exist_ok=True)

print('Feature:', FEATURE.upper())
print('Targets:', TARGET_SUBJECTS)
print('Device:', DEVICE_OBJ)
print('Output:', RUN_DIR)

Feature: RJSD
Targets: [1]
Device: cuda
Output: C:\Users\Lin\Documents\Arbitruam\CMRD-Cute-Mew-Really-Delighting\runs\diagnostics\seediv_feature_tuning\rjsd_plain_transformer_example_all_source_every10_diagnostic


## Replaceable model factory

The training loop only requires `model(data, mask) -> logits`, where `data` is `[batch, time, feature_dim]`, `mask` is `[batch, time]`, and `logits` is `[batch, 4]`. The implementation below is merely a runnable example; replace this cell when tuning another model.

In [3]:
def build_model(input_dim: int, classes: int, max_length: int) -> nn.Module:
    # Replace only this function to use another architecture.
    return PlainTransformer(
        input_dim=input_dim,
        classes=classes,
        max_length=max_length,
        **MODEL_CONFIG,
    )

## Data and metric helpers

Each fold uses all 14 non-target subjects (1008 trials) for training and the held-out target subject (72 trials) for testing. The RJSD reference, zDE statistics, and input normalization are recomputed from those same 14 source subjects only.

In [4]:
DATA_ROOT_PATH = feature_pipeline._resolve_data_root(DATA_ROOT)
CACHE_AUDIT = feature_pipeline.validate_cache(DATA_ROOT_PATH, deep=False)
print('Data root:', DATA_ROOT_PATH)
print('Cache:', CACHE_AUDIT['trials'], 'trials,', CACHE_AUDIT['folds'], 'folds')
print('ICA audit:', CACHE_AUDIT['ica'])
ALL_SOURCE_STATS_ROOT = ROOT / 'runs' / 'diagnostics' / 'seediv_feature_tuning' / '_all_source_statistics'
_PREPARED_FOLD_CACHE = {}  # Keeps only the most recently prepared fold in memory


def make_loader(samples, mean, std, shuffle: bool, seed: int) -> DataLoader:
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        SequenceDataset(samples, mean, std),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE_OBJ.type == 'cuda',
        collate_fn=collate_sequences,
        generator=generator,
    )


@torch.no_grad()
def evaluate_model(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> dict:
    model.eval()
    targets, predictions = [], []
    loss_sum = 0.0
    count = 0
    for data, mask, labels in loader:
        data = data.to(DEVICE_OBJ, non_blocking=True)
        mask = mask.to(DEVICE_OBJ, non_blocking=True)
        labels_device = labels.to(DEVICE_OBJ, non_blocking=True)
        logits = model(data, mask)
        loss = criterion(logits, labels_device)
        loss_sum += float(loss.item()) * labels.shape[0]
        count += labels.shape[0]
        targets.append(labels.numpy())
        predictions.append(logits.argmax(dim=1).cpu().numpy())
    metrics = classification_metrics(np.concatenate(targets), np.concatenate(predictions), classes=4)
    metrics['loss'] = loss_sum / count
    return metrics


def print_domain_metrics(name: str, metrics: dict) -> None:
    print(
        f"  {name}: loss={metrics['loss']:.4f} ACC={metrics['accuracy']:.4f} "
        f"BACC={metrics['balanced_accuracy']:.4f} Macro-F1={metrics['macro_f1']:.4f}"
    )
    for row in metrics['per_class']:
        print(
            f"    class={row['class']} P={row['precision']:.4f} R={row['recall']:.4f} "
            f"F1={row['f1']:.4f} support={row['support']}"
        )
    print(f"    confusion_matrix={metrics['confusion_matrix']}")


def fit_all_source_statistics(entries, need_de: bool, need_reference: bool):
    de_total = np.zeros((62, 5), dtype=np.float64) if need_de else None
    de_total_sq = np.zeros((62, 5), dtype=np.float64) if need_de else None
    reference_total = np.zeros((62, 5, 32), dtype=np.float64) if need_reference else None
    window_count = 0
    for entry in entries:
        with np.load(DATA_ROOT_PATH / entry['de_phist_path'], allow_pickle=False) as archive:
            if need_de:
                de = np.asarray(archive['de'], dtype=np.float64)
                de_total += de.sum(axis=0)
                de_total_sq += np.square(de).sum(axis=0)
            if need_reference:
                histogram = normalize_histograms(np.asarray(archive['p_hist'], dtype=np.float32))
                reference_total += histogram.sum(axis=0, dtype=np.float64)
            window_count += int(archive['de'].shape[0])
    de_mean = de_std = reference = None
    if need_de:
        de_mean = de_total / window_count
        variance = np.maximum(de_total_sq / window_count - np.square(de_mean), 0.0)
        de_std = np.sqrt(variance)
        de_std[de_std < 1e-6] = 1.0
        de_mean, de_std = de_mean.astype(np.float32), de_std.astype(np.float32)
    if need_reference:
        reference = normalize_histograms(reference_total / window_count)
    return de_mean, de_std, reference, window_count


def load_samples(entries, de_mean, de_std, reference):
    samples = []
    for entry in entries:
        need_de = FEATURE in {'de', 'cmrd', 'fusion'}
        need_rjsd = FEATURE in {'rjsd', 'cmrd', 'fusion'}
        de = rjsd = None
        with np.load(DATA_ROOT_PATH / entry['de_phist_path'], allow_pickle=False) as archive:
            if need_de:
                de = np.asarray(archive['de'], dtype=np.float32)
            if need_rjsd:
                histogram = np.asarray(archive['p_hist'], dtype=np.float32)
                flat = transform_rd(histogram, reference)
                rjsd = flat.reshape(flat.shape[0], 62, 5)
        if FEATURE == 'de':
            value = de
        elif FEATURE == 'rjsd':
            value = rjsd
        else:
            zde = (de - de_mean[None]) / de_std[None]
            cmrd = np.tanh(ALPHA * zde) * rjsd
            value = cmrd if FEATURE == 'cmrd' else np.concatenate((rjsd, cmrd, zde), axis=-1)
        samples.append(TrialSample(
            np.ascontiguousarray(value.reshape(value.shape[0], -1), dtype=np.float32),
            int(entry['label']), int(entry['subject']), int(entry['session']),
            int(entry['trial']), int(entry['source_index']),
        ))
    return samples


def prepare_fold(target_subject: int):
    cache_key = (int(target_subject), FEATURE, float(ALPHA), CACHE_AUDIT['preprocessing_signature'])
    if cache_key in _PREPARED_FOLD_CACHE:
        print(f'Reusing prepared fold {target_subject:02d} from memory')
        return _PREPARED_FOLD_CACHE[cache_key]
    fold_root = DATA_ROOT_PATH / 'folds' / f'fold-{target_subject:02d}'
    manifest = json.loads((fold_root / 'manifest.json').read_text(encoding='utf-8'))
    groups = manifest['groups']
    source_entries = sorted(groups['train'] + groups['validation'], key=lambda entry: int(entry['source_index']))
    target_entries = list(groups['test'])
    source_subjects = sorted({int(entry['subject']) for entry in source_entries})
    assert len(source_entries) == 1008 and len(source_subjects) == 14
    assert len(target_entries) == 72 and {int(entry['subject']) for entry in target_entries} == {target_subject}
    assert target_subject not in source_subjects

    need_de_stats = FEATURE in {'cmrd', 'fusion'}
    need_reference = FEATURE in {'rjsd', 'cmrd', 'fusion'}
    if need_de_stats or need_reference:
        stats_dir = ALL_SOURCE_STATS_ROOT / CACHE_AUDIT['preprocessing_signature']
        stats_dir.mkdir(parents=True, exist_ok=True)
        stats_path = stats_dir / f'fold-{target_subject:02d}.npz'
        if stats_path.is_file():
            with np.load(stats_path, allow_pickle=False) as archive:
                de_mean = np.asarray(archive['de_mean'], dtype=np.float32)
                de_std = np.asarray(archive['de_std'], dtype=np.float32)
                reference = np.asarray(archive['rjsd_reference'], dtype=np.float32)
                source_window_count = int(archive['source_window_count'])
                cached_subjects = list(map(int, archive['source_subjects']))
            if cached_subjects != source_subjects:
                raise RuntimeError(f'All-source statistics cache has wrong subjects: {stats_path}')
            print('Reusing all-source DE/RJSD statistics:', stats_path)
        else:
            print('Fitting all-source DE/RJSD statistics (first run for this fold) ...')
            de_mean, de_std, reference, source_window_count = fit_all_source_statistics(
                source_entries, need_de=True, need_reference=True
            )
            np.savez_compressed(
                stats_path, de_mean=de_mean, de_std=de_std, rjsd_reference=reference,
                source_window_count=np.int64(source_window_count), source_subjects=np.asarray(source_subjects),
            )
        if not need_de_stats:
            de_mean = de_std = None
        if not need_reference:
            reference = None
    else:
        de_mean = de_std = reference = None
        source_window_count = 0
    train_samples = load_samples(source_entries, de_mean, de_std, reference)
    target_samples = load_samples(target_entries, de_mean, de_std, reference)
    split = {'train_subjects': source_subjects, 'target_subject': target_subject, 'protocol': '14-source-train/1-target-test'}
    prepared = (train_samples, target_samples, split, de_mean, de_std, reference, source_window_count)
    _PREPARED_FOLD_CACHE.clear()
    _PREPARED_FOLD_CACHE[cache_key] = prepared
    return prepared

Data root: C:\Users\Lin\Documents\Arbitruam\Dataset\Processed\CMRD\seediv\de_rjsd_ica_1s_hop05\946b467993249ad9
Cache: 1080 trials, 15 folds
ICA audit: {'metadata_files': 1080, 'primary_fit_fallbacks': 8, 'detection_errors': 0, 'trials_with_zero_exclusions': 1, 'trials_with_cleaned_std_over_100_microvolt': 15}


## Generic per-epoch training loop

Training loss is printed every epoch. Every `TEST_EVERY` epochs, the notebook evaluates both the complete 14-subject source training domain and the held-out target subject, printing ACC/BACC/Macro-F1, per-class precision/recall/F1/support, and confusion matrices. Target metrics remain diagnostic outputs only.

In [5]:
SETTINGS = {
    'feature': FEATURE,
    'alpha': ALPHA,
    'targets': TARGET_SUBJECTS,
    'epochs': EPOCHS,
    'test_every': TEST_EVERY,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'minimum_learning_rate': MINIMUM_LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'label_smoothing': LABEL_SMOOTHING,
    'gradient_clip_norm': GRADIENT_CLIP_NORM,
    'split_protocol': '14-source-train/1-target-test',
    'seed': SEED,
    'deterministic': DETERMINISTIC,
    'device': DEVICE,
    'model_name': MODEL_NAME,
    'model_config': MODEL_CONFIG,
    'preprocessing_signature': CACHE_AUDIT['preprocessing_signature'],
    'diagnostic_target_peeking': True,
}
settings_path = RUN_DIR / 'settings.json'
if settings_path.is_file():
    previous = json.loads(settings_path.read_text(encoding='utf-8'))
    if previous != SETTINGS:
        raise RuntimeError('RUN_TAG already exists with different settings; change RUN_TAG')
else:
    settings_path.write_text(json.dumps(SETTINGS, indent=2, ensure_ascii=False), encoding='utf-8')


def train_one_fold(target_subject: int) -> pd.DataFrame:
    fold_dir = RUN_DIR / f'fold-{target_subject:02d}'
    fold_dir.mkdir(parents=True, exist_ok=True)
    epoch_csv = fold_dir / 'epochs.csv'
    evaluation_csv = fold_dir / 'evaluations.csv'
    if RESUME_COMPLETED_FOLDS and epoch_csv.is_file() and evaluation_csv.is_file():
        previous_epochs = pd.read_csv(epoch_csv)
        previous_evaluations = pd.read_csv(evaluation_csv)
        if len(previous_epochs) == EPOCHS and int(previous_epochs['epoch'].max()) == EPOCHS:
            print(f'fold={target_subject:02d}: reused {EPOCHS} completed epochs')
            return previous_evaluations

    seed_everything(SEED, DETERMINISTIC)
    train_samples, target_samples, split, de_mean, de_std, reference, source_window_count = prepare_fold(target_subject)
    normalizer_mean, normalizer_std = fit_normalizer(train_samples)
    train_loader = make_loader(train_samples, normalizer_mean, normalizer_std, True, SEED)
    source_eval_loader = make_loader(train_samples, normalizer_mean, normalizer_std, False, SEED)
    target_loader = make_loader(target_samples, normalizer_mean, normalizer_std, False, SEED)

    all_samples = train_samples + target_samples
    input_dim = train_samples[0].x.shape[1]
    max_length = max(sample.x.shape[0] for sample in all_samples)
    model = build_model(input_dim=input_dim, classes=4, max_length=max_length).to(DEVICE_OBJ)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=MINIMUM_LEARNING_RATE
    )

    epoch_rows = []
    evaluation_rows = []
    started = time.perf_counter()
    print(
        f'fold={target_subject:02d} feature={FEATURE.upper()} '
        f'train/test trials={len(train_samples)}/{len(target_samples)} '
        f'input_dim={input_dim} windows<= {max_length}'
    )

    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum = 0.0
        seen = 0
        for data, mask, labels in train_loader:
            data = data.to(DEVICE_OBJ, non_blocking=True)
            mask = mask.to(DEVICE_OBJ, non_blocking=True)
            labels = labels.to(DEVICE_OBJ, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(data, mask)
            loss = criterion(logits, labels)
            loss.backward()
            if GRADIENT_CLIP_NORM > 0:
                nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
            optimizer.step()
            loss_sum += float(loss.item()) * labels.shape[0]
            seen += labels.shape[0]
        scheduler.step()

        epoch_row = {
            'epoch': epoch,
            'train_loss': loss_sum / seen,
            'learning_rate': optimizer.param_groups[0]['lr'],
            'elapsed_seconds': time.perf_counter() - started,
        }
        epoch_rows.append(epoch_row)
        pd.DataFrame(epoch_rows).to_csv(epoch_csv, index=False)
        print(f"fold={target_subject:02d} epoch={epoch:03d}/{EPOCHS} train_loss={epoch_row['train_loss']:.4f}")

        if epoch % TEST_EVERY == 0 or epoch == EPOCHS:
            source_metrics = evaluate_model(model, source_eval_loader, criterion)
            target_metrics = evaluate_model(model, target_loader, criterion)
            evaluation_row = {
                'target_subject': target_subject,
                'feature': FEATURE,
                'epoch': epoch,
                'train_loss': epoch_row['train_loss'],
                'source_loss': source_metrics['loss'],
                'source_accuracy': source_metrics['accuracy'],
                'source_balanced_accuracy': source_metrics['balanced_accuracy'],
                'source_macro_f1': source_metrics['macro_f1'],
                'source_per_class': json.dumps(source_metrics['per_class'], separators=(',', ':')),
                'source_confusion_matrix': json.dumps(source_metrics['confusion_matrix'], separators=(',', ':')),
                'target_loss': target_metrics['loss'],
                'target_accuracy': target_metrics['accuracy'],
                'target_balanced_accuracy': target_metrics['balanced_accuracy'],
                'target_macro_f1': target_metrics['macro_f1'],
                'target_per_class': json.dumps(target_metrics['per_class'], separators=(',', ':')),
                'target_confusion_matrix': json.dumps(target_metrics['confusion_matrix'], separators=(',', ':')),
                'elapsed_seconds': epoch_row['elapsed_seconds'],
            }
            evaluation_rows.append(evaluation_row)
            pd.DataFrame(evaluation_rows).to_csv(evaluation_csv, index=False)
            print(f'--- diagnostic evaluation: fold={target_subject:02d} epoch={epoch:03d} ---')
            print_domain_metrics('SOURCE', source_metrics)
            print_domain_metrics('TARGET', target_metrics)

    if SAVE_FINAL_CHECKPOINT:
        torch.save(
            {
                'model_state_dict': {key: value.detach().cpu() for key, value in model.state_dict().items()},
                'normalization_mean': normalizer_mean,
                'normalization_std': normalizer_std,
                'de_mean': de_mean,
                'de_std': de_std,
                'rjsd_reference': reference,
                'source_train_windows': source_window_count,
                'final_epoch': int(epoch_rows[-1]['epoch']),
                'split': split,
                'settings': SETTINGS,
                'warning': 'Target was evaluated every TEST_EVERY epochs; diagnostic only.',
            },
            fold_dir / 'final_epoch.pt',
        )

    frame = pd.DataFrame(evaluation_rows)
    del model, optimizer, scheduler, train_loader, source_eval_loader, target_loader
    del train_samples, target_samples, all_samples
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return frame

## Run selected LOSO folds

The table is refreshed after every completed fold. `target_*` columns are deliberately diagnostic and target-peeked.

In [6]:
fold_frames = []
for target_subject in TARGET_SUBJECTS:
    fold_frame = train_one_fold(target_subject)
    fold_frames.append(fold_frame)
    display(fold_frame.tail(1))

all_epochs = pd.concat(fold_frames, ignore_index=True)
all_epochs.to_csv(RUN_DIR / 'all_fold_epochs.csv', index=False)
epoch_summary = (
    all_epochs.groupby('epoch', as_index=False)
    .agg(
        source_accuracy_mean=('source_accuracy', 'mean'),
        source_macro_f1_mean=('source_macro_f1', 'mean'),
        target_accuracy_mean=('target_accuracy', 'mean'),
        target_accuracy_std=('target_accuracy', lambda values: values.std(ddof=0)),
        target_macro_f1_mean=('target_macro_f1', 'mean'),
        target_macro_f1_std=('target_macro_f1', lambda values: values.std(ddof=0)),
        subjects=('target_subject', 'nunique'),
    )
)
epoch_summary.to_csv(RUN_DIR / 'epoch_summary.csv', index=False)
display(epoch_summary.tail(20).style.format({
    'source_accuracy_mean': '{:.2%}',
    'source_macro_f1_mean': '{:.2%}',
    'target_accuracy_mean': '{:.2%}',
    'target_accuracy_std': '{:.2%}',
    'target_macro_f1_mean': '{:.2%}',
    'target_macro_f1_std': '{:.2%}',
}))

Reusing all-source DE/RJSD statistics: C:\Users\Lin\Documents\Arbitruam\CMRD-Cute-Mew-Really-Delighting\runs\diagnostics\seediv_feature_tuning\_all_source_statistics\946b467993249ad9\fold-01.npz
fold=01 feature=RJSD train/test trials=1008/72 input_dim=310 windows<= 517
fold=01 epoch=001/100 train_loss=1.4518
fold=01 epoch=002/100 train_loss=1.3717
fold=01 epoch=003/100 train_loss=1.2744
fold=01 epoch=004/100 train_loss=1.1917
fold=01 epoch=005/100 train_loss=1.1265
fold=01 epoch=006/100 train_loss=0.9967
fold=01 epoch=007/100 train_loss=0.8968
fold=01 epoch=008/100 train_loss=0.7968
fold=01 epoch=009/100 train_loss=0.7116
fold=01 epoch=010/100 train_loss=0.6750
--- diagnostic evaluation: fold=01 epoch=010 ---
  SOURCE: loss=0.6506 ACC=0.9683 BACC=0.9683 Macro-F1=0.9680
    class=0 P=0.9826 R=0.8968 F1=0.9378 support=252
    class=1 P=0.9197 R=1.0000 F1=0.9582 support=252
    class=2 P=0.9920 R=0.9841 F1=0.9880 support=252
    class=3 P=0.9843 R=0.9921 F1=0.9881 support=252
    confusio

KeyboardInterrupt: 

In [ ]:
# Optional diagnostic curves.
try:
    import matplotlib.pyplot as plt

    figure, axis = plt.subplots(figsize=(10, 5))
    axis.plot(epoch_summary['epoch'], epoch_summary['source_accuracy_mean'], '--', label='Source ACC')
    axis.plot(epoch_summary['epoch'], epoch_summary['target_accuracy_mean'], label='Target ACC (peeked)')
    axis.fill_between(
        epoch_summary['epoch'],
        epoch_summary['target_accuracy_mean'] - epoch_summary['target_accuracy_std'],
        epoch_summary['target_accuracy_mean'] + epoch_summary['target_accuracy_std'],
        alpha=0.2,
    )
    axis.set(xlabel='Epoch', ylabel='Accuracy', title=f'SEED-IV {FEATURE.upper()} diagnostic target monitoring')
    axis.grid(alpha=0.3)
    axis.legend()
    figure.tight_layout()
    figure.savefig(RUN_DIR / 'accuracy_curve.png', dpi=160)
    plt.show()
except ModuleNotFoundError:
    print('matplotlib is unavailable; CSV results were still saved')

## Interpretation

- Every non-target subject is included in training; there is no source-validation holdout.
- Target curves are useful for diagnosing domain shift, collapse, and unstable epochs.
- Do not report the best target epoch from this notebook as a clean final result. A formal experiment must pre-fix the epoch count/hyperparameters or use a separate nested validation protocol.